<h1 style = "color : #0EE071; text-align : center;"><em>Where should I live?</em> - Data Science in Action Notebook</h1>
<p style = "font-size : 16px; text-align: center;">
In this final phase, you will bring everything together by integrating your preprocessed dataset and transforming it into <code>meaningful insights or tools</code>. This is
your opportunity to be creative: choose techniques you find most appropriate and
develop something that would <code>genuinely help people compare European cities
and decide where to live</code>.</p>
<br>
<p style = "font-size : 12px; text-align: center;"><b>NOVA IMS</b></p>
<p style = "font-size : 10px; text-align: center;">Programming for Data Science</p>
<p style = "font-size : 10px; text-align: center;">Diogo Gonçalves, João Marques, Juan Mendes & Gustavo Franco</p>
<br>

<h2  style = "color : #0EE071;"> Imports</h2>

In [2]:
#!pip install panel

In [2]:
from selenium import webdriver
from selenium.webdriver.common.by import By
import time
from bs4 import BeautifulSoup
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.keys import Keys
import requests
import re
import pandas as pd
import panel as pn
import time
import plotly.express as px
import numpy as np
import panel as pn
import pandas as pd
import plotly.graph_objects as go
import warnings
warnings.filterwarnings('ignore')

<hr style = "border: 3px solid #0EE071;">
<h2 style = "color : #0EE071;">Dataset Importing </h2>
<p style = "font-size : 15px;">Reading of dataset from <code>city_data_clean.csv</code> file</p>
<p>The dataset is in a <code>.csv</code> file, and uses <code>,</code> as a separator.</p>

In [46]:
city_data = pd.read_csv("city_data_clean.csv", sep = ",")

FileNotFoundError: [Errno 2] No such file or directory: 'city_data_clean.csv'

<hr style = "border: 3px solid #0EE071;">
<h2  style = "color : #0EE071;">Web Scraping</h2>
<p style="font-size: 15px;">
  <span style="font-size: 20px;">Short summary:</span>
  <br><br>
  - <code>requests</code> : download HTML only<br>
  - <code>BeautifulSoup</code> : extract info from that HTML<br>
  - <code>Selenium</code> : control a browser and interact with the page
</p>

In [105]:
url_dictionary = {"Food Prices": "https://www.numbeo.com/food-prices/in/city",
"Gas Prices Calculator" : "https://www.numbeo.com/gas-prices/in/city",
"Salary Calculator" : "https://www.numbeo.com/cost-of-living/prices_by_country.jsp?itemId=105&displayCurrency=EUR",
"Quality of Life": "https://www.numbeo.com/quality-of-life/in/city",
"Crime": "https://www.numbeo.com/crime/in/city",
"Pollution": "https://www.numbeo.com/pollution/in/city"}

for element in url_dictionary:
    city_data.loc[:, element] = None

In [ ]:
def food_prices(readable_html):
    rows = readable_html.find_all("tr", class_=["tr_standard", "tr_highlighted"])
    #data = []
    total = 0
    for row in rows:
        tds = row.find_all("td")
        #item = tds[0].get_text(strip=True).replace("(", "").replace(")", "")
        price = tds[1].get_text(strip=True).replace("\xa0", "").replace("(", "").replace(")", "")
        #data.append([item, price])
        number = float(re.search(r"\d+[\.,]?\d*", price).group().replace(',', '.'))
        total += number 
    return total

def gas_prices(readable_html):
    value = readable_html.find("span", class_="first_currency").text
    number = float(re.search(r"\d+[\.,]?\d*", value).group().replace(',', '.'))
    return number

def quality_of_life(readable_html):
    total = 0
    for element in readable_html.find_all("td", style="text-align: right"):
        text = ''.join(digit for digit in element.get_text(strip=True) if digit.isdigit() or digit == '.')
        if text:
            total += float(text)
    return total

def crime(readable_html):
    total = 0
    for element in readable_html.find_all("td", style="text-align: right"):
        text = ''.join(digit for digit in element.get_text(strip=True) if digit.isdigit() or digit == '.')
        if text:
            total += float(text)
    return total

def pollution(readable_html):
    total = 0
    for element in readable_html.find_all("td", style="text-align: right"):
        text = ''.join(digit for digit in element.get_text(strip=True) if digit.isdigit() or digit == '.')
        if text:
            total += float(text)
    return total

def salary(readable_html, country):
    li = [x.strip() for x in readable_html.body.text.split("(After Tax)")[4].split("Last Update")[0].split("\n") if x.strip() != ""] 
    for item in li:
        if country in item:
            number = float(re.search(r"\d+\.\d+", item).group())
    return number
    
def get_info(data):
    for index in data.index:
        city_name = data.loc[index, "City"]
        if city_name == "Bruges":
            city_name = "Brugge"
        if city_name == "Lefkosia":
            city_name = "Nicosia"
        if city_name == "Lemesos":
            city_name = "Limassol"
        if city_name == "Frankfurt am Main":
            city_name = "Frankfurt"
        if city_name == "Seville":
            city_name = "Sevilla"
        if city_name == "The Hague":
            city_name = "The-Hague-Den-Haag-Netherlands"
        if city_name == "Cracow":
            city_name = "Krakow-Cracow"
        if city_name == "Cracow":
            city_name = "Krakow-Cracow"
        
        country_name = data.loc[index, "Country"]
        for char in url_dictionary:
            url = url_dictionary[char].replace("city", city_name)
            html = requests.get(url)
            readable_html = BeautifulSoup(html.text, "html.parser")

            if char == "Food Prices":
                try:
                    data.loc[index, char] = food_prices(readable_html)

                except:
                    data.loc[index, char] = 0

            elif char == "Gas Prices Calculator":
                try:
                    data.loc[index, char] = gas_prices(readable_html)

                except:
                    data.loc[index, char] = 0

            elif char == "Salary Calculator":
                try:
                    data.loc[index, char] = salary(readable_html, country_name)
                except:
                    data.loc[index, char] = 0
            elif char == "Quality of Life":
                try:
                    data.loc[index, char] = quality_of_life(readable_html)
                except:
                    data.loc[index, char] = 0

            elif char == "Crime":
                try:
                    data.loc[index, char] = crime(readable_html)

                except:
                    data.loc[index, char] = 0
            elif char == "Pollution":
                try:
                    data.loc[index, char] = pollution(readable_html)
                except:
                    data.loc[index, char] = 0

In [114]:
get_info(city_data)

In [112]:
city_data[['City', 'Country', 'Food Prices', 'Gas Prices Calculator', 'Salary Calculator', 'Quality of Life', 'Crime', 'Pollution']]

,City,Country,Food Prices,Gas Prices Calculator,Salary Calculator,Quality of Life,Crime,Pollution
0,Vienna,Austria,22.540000000000003,1.55,2662.69,911.66,625.6800000000001,874.3000000000001
1,Salzburg,Austria,23.899999999999995,1.5,2662.69,872.63,545.52,891.73
2,Brussels,Belgium,20.480000000000004,1.64,2612.9,816.1199999999999,923.2200000000001,1016.64
3,Antwerp,Belgium,18.849999999999998,1.63,2612.9,848.5500000000001,753.1499999999999,1000.1400000000001
4,Gent,Belgium,19.169999999999998,1.61,2612.9,911.58,587.11,901.62
...,...,...,...,...,...,...,...,...
79,Stockholm,Sweden,235.67000000000004,17.16,2804.9,800.43,814.14,871.41
80,Gothenburg,Sweden,216.21,17.59,2804.9,884.96,801.3,876.0400000000001
81,Malmo,Sweden,232.77000000000004,18.08,2804.9,826.8300000000002,904.23,862.94
82,Ankara,Turkiye,580.7299999999999,51.16,0,717.8,747.07,1062.28


In [95]:
url = "https://www.numbeo.com/cost-of-living/prices_by_country.jsp?itemId=105&displayCurrency=EUR"
html = requests.get(url)
readable_html = BeautifulSoup(html.text, "html.parser")
li = [x.strip() for x in readable_html.body.text.split("(After Tax)")[4].split("Last Update")[0].split("\n") if x.strip() != ""] 
for item in li:
    if "Portugal" in item:
        print(item)
        number = float(re.search(r"\d+\.\d+", item).group())
        print(number)

Portugal1117.69
1117.69


<hr style = "border: 3px solid #0EE071;">
<h2 style = "color : #0EE071;">Dashboard</h2>
<p style = "font-size : 16px;">It displays our data interactively through a dashboard</p>
<br>

Import of Images

In [115]:
country_flag_urls = {
    "Austria": "https://flagcdn.com/w160/at.png",
    "Belgium": "https://flagcdn.com/w160/be.png",
    "Bulgaria": "https://flagcdn.com/w160/bg.png",
    "Switzerland": "https://flagcdn.com/w160/ch.png",
    "Cyprus": "https://flagcdn.com/w160/cy.png",
    "Czechia": "https://flagcdn.com/w160/cz.png",
    "Germany": "https://flagcdn.com/w160/de.png",
    "Denmark": "https://flagcdn.com/w160/dk.png",
    "Spain": "https://flagcdn.com/w160/es.png",
    "Estonia": "https://flagcdn.com/w160/ee.png",
    "Finland": "https://flagcdn.com/w160/fi.png",
    "France": "https://flagcdn.com/w160/fr.png",
    "United Kingdom": "https://flagcdn.com/w160/gb.png",
    "Greece": "https://flagcdn.com/w160/gr.png",
    "Croatia": "https://flagcdn.com/w160/hr.png",
    "Hungary": "https://flagcdn.com/w160/hu.png",
    "Ireland": "https://flagcdn.com/w160/ie.png",
    "Italy": "https://flagcdn.com/w160/it.png",
    "Luxembourg": "https://flagcdn.com/w160/lu.png",
    "Latvia": "https://flagcdn.com/w160/lv.png",
    "Malta": "https://flagcdn.com/w160/mt.png",
    "Netherlands": "https://flagcdn.com/w160/nl.png",
    "Norway": "https://flagcdn.com/w160/no.png",
    "Poland": "https://flagcdn.com/w160/pl.png",
    "Portugal": "https://flagcdn.com/w160/pt.png",
    "Romania": "https://flagcdn.com/w160/ro.png",
    "Slovak Republic": "https://flagcdn.com/w160/sk.png",
    "Slovenia": "https://flagcdn.com/w160/si.png",
    "Sweden": "https://flagcdn.com/w160/se.png",
    "Turkiye": "https://flagcdn.com/w160/tr.png"
}

In [116]:
city_data["Image Url"] = None
for index in city_data.index:
    for country, url in country_flag_urls.items():
        if country in city_data.loc[index, "Country"]:
            city_data.loc[index, "Image Url"] = url
            break

In [ ]:
#city_data.loc[36, "Country"]

' France'

Dashboard

In [117]:
cols_needed = ['Population Density', 'Population', 'Working Age Population',
       'Youth Dependency Ratio', 'Unemployment Rate', 'GDP per Capita',
       'Days of very strong heat stress', 'Average Monthly Salary',
       'Average Rent Price', 'Average Cost of Living',
       'Food Prices', 'Gas Prices Calculator', 'Salary Calculator',
       'Quality of Life', 'Crime', 'Pollution']

In [148]:
# Initialize Panel extension
pn.extension('plotly')

# ======================================
# DATA PREPARATION
# ======================================
# Clean column names and fill NA values
city_data.columns = [col.strip() for col in city_data.columns]
city_data = city_data.fillna(0)

# ======================================
# STYLES
# ======================================
dashboard_style = {'background': '#F7F9FC', 'padding': '25px', 'border-radius': '15px'}
header_style = {'background': '#2A3B4D', 'padding': '5px 25px', 'border-radius': '10px', 'margin': '0 0 20px 0'}
card_style = {'background': '#FFFFFF', 'border': '1px solid #E0E0E0', 'border-radius': '10px', 'padding': '25px', 'margin': '10px 0'}
slider_style = {'background': '#FFFFFF', 'border-radius': '8px', 'padding': '15px', 'margin': '8px 0', 'box-shadow': '0 2px 12px rgba(0, 0, 0, 0.05)', 'border-left': '4px solid #0D6EFD',  'border-top': '1px solid #E0E0E0', 'border-right': '1px solid #E0E0E0', 'border-bottom': '1px solid #E0E0E0'}
infobox_style = {'background': '#FFFFFF', 'border-left': '5px solid #0D6EFD', 'border-radius': '8px', 'padding': '10px', 'margin': '8px', 'box-shadow': '0 2px 6px 0 rgba(0,0,0,0.07)', 'width': '260px', 'height': '80px', 'box-sizing': 'border-box', 'text-align': 'center', 'display': 'flex', 'flex-direction': 'column', 'justify-content': 'center'}

# ======================================
# WIDGETS
# ======================================
# City selectors
country_1 = pn.widgets.Select( name="City", options=sorted(list(city_data["City"])), min_width=200, width_policy='max', sizing_mode='stretch_width')
country_2 = pn.widgets.Select(name="City", options=list(city_data["City"]), min_width=200, width_policy='max', sizing_mode='stretch_width')

# View selector
select = pn.widgets.RadioButtonGroup( name="Select", options=["Characteristics", "Compare Cities"],  button_type='primary', button_style='outline', min_width=200, sizing_mode='stretch_width')

# Navigation buttons
big_button = pn.widgets.Button(name="Your Country Is...", button_type='primary', height=40, styles={ 'font-size': '16px', 'font-weight': 'bold', 'padding': '5px'}, sizing_mode='stretch_width', width_policy='max')
go_back_button = pn.widgets.Button( name='Go Back', button_type='default', sizing_mode='stretch_width')

# ======================================
# DATA PROCESSING FUNCTIONS
# ======================================
def scale(data, cols):
    """Scale numeric columns for comparison"""
    for col in cols:
        data[f"{col} Scaled"] = data[col] / data[col].sum()

# Apply scaling
scale(city_data, cols_needed)

def best_country(sliders, data):
    """Calculate best matching country based on slider weights"""
    final = pd.Series(0, index=data["City"], name="Scores")
    for index in final.index:
        total = 0
        for name, slider in sliders.items():
            col = f"{name} Scaled"
            total += slider.value * data.loc[data["City"] == index, col].iloc[0]
        final[index] = total
    final = final.sort_values(ascending=False)
    return final.idxmax(), final

def update_display(*_):
    """Update display when sliders change"""
    best, final = best_country(sliders, city_data)
    return best, final

# ======================================
# UI COMPONENTS
# ======================================
def binary(option):
    """Render either single city or comparison view based on selection"""
    def get_image(city):
        return pn.pane.PNG(city_data[city_data["City"] == city].iloc[0]["Image Url"], width=240, height=160)

    interative_image = pn.bind(get_image, country_1)  
    
    if option == "Characteristics":
        return pn.Column(pn.pane.Markdown("### Your most wanted country"), interative_image, styles=card_style)
    else:
        return pn.Column(
            pn.pane.Markdown("### Choose your country"), 
            pn.Column(country_1, country_2), 
            styles=card_style
        )

def chars_left(selected_chars, city):
    """Display selected characteristics for a country"""
    global boards
    boards = []
    
    emoji_map = {
    "Population Density": "👨‍👩‍👧",
    "Population": "🌎",
    "Working Age Population": "💼🔞",
    "Youth Dependency Ratio": "👶🏻",
    "Unemployment Rate": "💼📉",
    "GDP per Capita": "💰",
    "Days of very strong heat stress": "🌡️🔥",
    "Average Monthly Salary": "💼💰",
    "Average Rent Price": "🏠💲",
    "Average Cost of Living": "💸👨‍👨‍👧‍👦",
    "Food Prices": "🥦💲",
    "Gas Prices Calculator": "⛽💲",
    "Salary Calculator": "📊💼",
    "Quality of Life": "🌿😊",
    "Crime": "🚔⚠️",
    "Pollution": "🏭😷"
}

    
    for char in selected_chars:
        if char in emoji_map:
            emoji = emoji_map[char]
            board = pn.pane.Markdown(f"<div style='font-size:8pt'><b>{char}</b> {emoji}</div><br><div style='font-size:7pt'>Current value: <b>{round(city_data[city_data["City"] == city].iloc[0][char], 0)}</b></div>", styles=infobox_style)
            boards.append(board)

    # Create rows of 3 columns
    rows = []
    for i in range(0, len(boards), 3):
        rows.append(pn.Row(*boards[i:i+3]))
    
    return pn.Column(*rows)

def all_chars(chars, city_1, city_2):
    """Display comparison of all characteristics between two countries"""
    global boards
    boards = []
    
    emoji_map = {
    "Population Density": "👨‍👩‍👧",
    "Population": "🌎",
    "Working Age Population": "💼🔞",
    "Youth Dependency Ratio": "👶🏻",
    "Unemployment Rate": "💼📉",
    "GDP per Capita": "💰",
    "Days of very strong heat stress": "🌡️🔥",
    "Average Monthly Salary": "💼💰",
    "Average Rent Price": "🏠💲",
    "Average Cost of Living": "💸👨‍👨‍👧‍👦",
    "Food Prices": "🥦💲",
    "Gas Prices Calculator": "⛽💲",
    "Salary Calculator": "📊💼",
    "Quality of Life": "🌿😊",
    "Crime": "🚔⚠️",
    "Pollution": "🏭😷"
}

    
    # Get image URLs
    img1_url = city_data[city_data["City"] == city_1].iloc[0]["Image Url"]
    img2_url = city_data[city_data["City"] == city_2].iloc[0]["Image Url"]
    
    for char in chars:
        if char in emoji_map:
            # Create image panes with proper HTML
            image1 = pn.pane.HTML(f'<img src="{img1_url}" width="30" height="20" style="vertical-align:middle">')
            image2 = pn.pane.HTML(f'<img src="{img2_url}" width="30" height="20" style="vertical-align:middle">')
            emoji = emoji_map[char]
            
            comparison_row = pn.Row(
                image1,
                pn.pane.HTML(f"<div style='text-align:center; font-size:9px'><b>{round(city_data[city_data["City"] == city_1].iloc[0][char], 0)}</b></div>"),
                pn.pane.HTML("<div style='text-align:center; font-size:10px'><b>vs</b></div>"),
                pn.pane.HTML(f"<div style='text-align:center; font-size:9px'><b>{round(city_data[city_data["City"] == city_2].iloc[0][char], 0)}</b></div>"),
                image2, align='center')
        
            board = pn.Column(pn.Column(pn.pane.Markdown(f"<div style='text-align:center; font-size:12px'><b>{char} {emoji}</b></div>"), align='center'), comparison_row, styles=infobox_style, align='center')
            
            boards.append(board)

    # Create rows of 3 columns
    rows = []
    for i in range(0, len(boards), 3):
        rows.append(pn.Row(*boards[i:i+3]))
    
    return pn.Column(*rows)

def create_pie_chart(*values):
    """Create a pie chart showing weight distribution"""
    all_labels = list(sliders.keys())
    values_list = []
    labels_list = []
    
    for i in range(len(values)):
        if values[i] > 0:
            values_list.append(values[i])
            labels_list.append(all_labels[i])
    
    if not values_list:
        return pn.pane.Markdown("### Adjust the sliders to see the weight distribution")

    n = len(values_list)
    blue_colors = [f'rgb({int(30 + i * (225/n))}, {int(100 + i * (155/n))}, {int(180 + i * (75/n))})' for i in range(n)]
    
    pie = go.Pie(labels=labels_list, values=values_list, textinfo='label+percent', hole=0.3, textfont_size=12, textposition='inside', marker_colors=blue_colors)
    
    fig = go.Figure(data=[pie])
    
    fig.update_layout(margin=dict(l=10, r=10, t=30, b=10), showlegend=False, height=250, width=400, xaxis=dict(showgrid=False, zeroline=False), yaxis=dict(showgrid=False, zeroline=False))
    
    return pn.pane.Plotly(fig, config={'displayModeBar': False}, width=400, height=250, sizing_mode='fixed')

# ======================================
# SLIDERS CREATION
# ======================================
sliders = {}
for col in cols_needed:
    sliders[col] = pn.widgets.IntSlider(name=col, start=0, end=10, step=1, width=200, styles=slider_style, width_policy='max')

# Create 3 columns for sliders
slider_columns = [[], [], []]
for i, (col, slider) in enumerate(sliders.items()):
    column_index = i % 3
    slider_columns[column_index].append(slider)

# Create Rows for each column of sliders
slider_rows = []
for col_sliders in slider_columns:
    if col_sliders:
        slider_rows.append(pn.Column(*col_sliders, width=190))

# Create sliders panel
sliders_panel = pn.Column(pn.pane.Markdown("### On a scale from one to ten, how important are these characteristics?", styles={'color': '#0D6EFD'}), pn.Row(*slider_rows), styles={'background': '#FFFFFF', 'padding': '15px', 'border-radius': '8px'}, width_policy='max')

# Initialize slider values
slider_values = [slider.param.value for slider in sliders.values()]

# Create the pie chart panel
pie_chart = pn.panel(pn.bind(create_pie_chart, *slider_values), config={'displayModeBar': False})

# ======================================
# LAYOUTS
# ======================================
header = pn.Row(pn.pane.Markdown("## Dashboard", styles={'color': '#FFFFFF'}), styles=header_style, sizing_mode='stretch_width')

def one_country():
    """Layout for single country view"""
    header = pn.Row(pn.pane.Markdown("## Dashboard", styles={'color': '#FFFFFF'}), styles=header_style, sizing_mode='stretch_width')
    under_header = pn.Row(select, styles=card_style, width=413)
    left = pn.Column(pn.panel(interactive_binary), styles=card_style, width=373, sizing_mode='fixed')
    return pn.Column( header, pn.Row(pn.Column(under_header, pie_chart), pn.Spacer(width = 150) , sliders_panel), pn.Row(big_button, styles={'background': '#2A3B4D', 'padding': '5px 25px', 'border-radius': '10px', 'margin': '0 0 10px 0'}, sizing_mode='stretch_width'))

def two_countries():
    """Layout for comparing two countries"""
    header = pn.Row(pn.pane.Markdown("## Dashboard", styles={'color': '#FFFFFF'}), styles=header_style, sizing_mode='stretch_width')
    under_header = pn.Row(select, styles=card_style)
    return pn.Column(header, pn.Row(pn.Column(under_header, interactive_binary, sizing_mode='fixed'), pn.Spacer(width = 75) ,interactive_boards_all, styles=card_style))

def layout_choice(selection):
    """Choose between single country or comparison layout"""
    if selection == "Characteristics":
        return one_country()
    else:
        return two_countries()

# ======================================
# INTERACTIVE BINDINGS
# ======================================
# Interaction between the select "Characteristics" and "Comparison of the two cities"
interactive_binary = pn.bind(binary, select)

# Comparison of the layout choice function and the select box
interactive_layout = pn.bind(layout_choice, select)

# Get all characteristics for comparison
chars = cols_needed

# Bind the all_chars function to city selectors
interactive_boards_all = pn.bind(lambda city1, city2: all_chars(chars, city1, city2), country_1.param.value, country_2.param.value)

# ======================================
# PAGES
# ======================================
def page1():
    """Results page showing best matching country"""
    
    best, scores = best_country(sliders, city_data)
    #print(best, scores)
    best_score = scores[best]
    #print(best_score)

    chars = cols_needed

    results = chars_left(chars, best)
    #print(results)
    def get_image(_):
        return pn.pane.PNG(city_data[city_data["City"] == best].iloc[0]["Image Url"], width=240, height=160)
    
    return pn.Column(header,pn.Row(pn.Column( pn.pane.Markdown("## Your Perfect Match"), pn.bind(get_image, None), pn.pane.Markdown(f"### {best}"), results), pn.Spacer(width = 100) ,pn.Column(pn.pane.Markdown(f"**Match Score:** {best_score:.2f}"), pn.pane.Markdown("### All Scores:"), pn.pane.DataFrame(scores.to_frame("Score").head(26)))), go_back_button, margin=20)

# ======================================
# NAVIGATION
# ======================================
def show_page(page_name, event=None):
    """Handle page navigation"""
    dashboard.clear()
    
    if page_name == 'home':
        # Always check the current select value
        selection = select.value
        if selection == "Characteristics":
            page = one_country()
        else:
            page = two_countries()
        big_button.on_click(lambda e: show_page('page1'))
        dashboard.append(page)
        
    elif page_name == 'page1':
        page = page1()
        go_back_button.on_click(lambda e: show_page('home'))
        dashboard.append(page)

# ======================================
# INITIALIZATION
# ======================================
# Create dashboard
dashboard = pn.Column(styles=dashboard_style, sizing_mode='stretch_width')

# Set up view selector callback
def on_select_change(event):
    show_page('home')

select.param.watch(on_select_change, 'value')

# Bind the best country display
best_country_display = pn.bind(update_display, *[s.param.value for s in sliders.values()])

# Start with home page
show_page('home')

# ======================================
# SERVE THE DASHBOARD
# ======================================
pn.serve(dashboard)

Launching server at http://localhost:63526


In [134]:
city_data

,Population Density,Population,Working Age Population,Youth Dependency Ratio,Unemployment Rate,GDP per Capita,Days of very strong heat stress,Average Monthly Salary,Average Rent Price,Average Cost of Living,...,Days of very strong heat stress Scaled,Average Monthly Salary Scaled,Average Rent Price Scaled,Average Cost of Living Scaled,Food Prices Scaled,Gas Prices Calculator Scaled,Salary Calculator Scaled,Quality of Life Scaled,Crime Scaled,Pollution Scaled
0,310.0,2983513,2018818,20.1,10.2,55770.0,3,2500,1050,2061,...,0.008086,0.012257,0.011732,0.013566,0.001094,0.000694,0.014032,0.013512,0.010658,0.011340
1,243.0,375489,250472,20.3,3.0,66689.0,0,3200,1100,2186,...,0.000000,0.015689,0.012291,0.014389,0.001160,0.000672,0.014032,0.012933,0.009292,0.011566
2,681.0,3284548,2137425,27.5,10.7,62500.0,3,3350,1200,1900,...,0.008086,0.016424,0.013408,0.012507,0.000994,0.000735,0.013769,0.012096,0.015726,0.013186
3,928.0,1139663,723396,27.7,6.2,57595.0,3,2609,900,1953,...,0.008086,0.012791,0.010056,0.012855,0.000915,0.000730,0.013769,0.012576,0.012829,0.012972
4,552.0,645813,417832,24.8,5.3,53311.0,2,2400,827,1200,...,0.005391,0.011767,0.009241,0.007899,0.000930,0.000721,0.013769,0.013510,0.010001,0.011694
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
79,334.0,2344124,1534225,28.5,6.2,70950.0,0,2700,1400,2300,...,0.000000,0.013238,0.015643,0.015139,0.011434,0.007688,0.014781,0.011863,0.013868,0.011302
80,245.0,1037675,672152,28.2,6.3,49588.0,0,2500,1200,2100,...,0.000000,0.012257,0.013408,0.013823,0.010490,0.007880,0.014781,0.013116,0.013649,0.011362
81,368.0,680335,436271,29.4,9.2,44387.0,0,2400,1100,2000,...,0.000000,0.011767,0.012291,0.013165,0.011294,0.008100,0.014781,0.012254,0.015403,0.011192
82,1922.0,4843511,3417691,30.0,14.4,38916.0,3,900,450,900,...,0.008086,0.004413,0.005028,0.005924,0.028176,0.022920,0.000000,0.010638,0.012726,0.013778
